# Stage 1 — Neo4j GDS / tabular baseline

This notebook trains the memory-safe Stage 1 baseline on the PRING pair table. It works with either a full PRING run folder, `<run>/graph/ml/modeling`, or the standalone attached `stage1_neo4j_gds_baselines` folder.


In [ ]:
# Make the local modeling package importable when running notebooks from VS Code/Jupyter.
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
MODELING_ROOT = None

for candidate in [HERE, *HERE.parents]:
    if (candidate / "pring_modeling").is_dir():
        MODELING_ROOT = candidate
        break
    if (candidate / "modeling" / "pring_modeling").is_dir():
        MODELING_ROOT = candidate / "modeling"
        break

if MODELING_ROOT is None:
    raise RuntimeError(
        "Could not find the local pring_modeling package. "
        "Open this notebook from the project/modeling folder, or run: pip install -e ./modeling"
    )

if str(MODELING_ROOT) not in sys.path:
    sys.path.insert(0, str(MODELING_ROOT))

print(f"Using pring_modeling from: {MODELING_ROOT}")


In [ ]:
from pathlib import Path

# Change this path when running locally or inside Docker.
MODEL_INPUT = Path("A:/Repositories/PRING/runs/cyp450_5enzymes_uncapped_raw_rematerialized/graph/ml/modeling/stage1_neo4j_gds_baselines")
OUTPUT_DIR = Path("/models/notebook_stage1_tabular")
REPORT_DIR = Path("/reports/modeling")

print(f"MODEL_INPUT = {MODEL_INPUT}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"REPORT_DIR  = {REPORT_DIR}")


In [ ]:
from pring_modeling.stage1_tabular import build_parser, run

args = build_parser().parse_args([
    "--modeling-dir", str(MODEL_INPUT),
    "--output-dir", str(OUTPUT_DIR),
    "--report-dir", str(REPORT_DIR),
    "--max-training-rows", "100000",
    "--max-scoring-rows", "100000",
    "--n-estimators", "200",
])
summary = run(args)
summary


In [ ]:
import pandas as pd

pred = pd.read_csv(OUTPUT_DIR / "predictions.csv")
pred.head(20)


## Optional: train MLP on exported GDS embeddings

Run this only after exporting FastRP/GraphSAGE embeddings from Neo4j GDS to a CSV containing `node_id`/`node_ref` and `embedding`.


In [ ]:
# from pring_modeling.stage1_mlp_gds import build_parser as mlp_parser, run as run_mlp
# EMBEDDING_CSV = Path("/models/gds_embeddings.csv")
# args = mlp_parser().parse_args([
#     "--modeling-dir", str(MODEL_INPUT),
#     "--embedding-csv", str(EMBEDDING_CSV),
#     "--output-dir", "/models/notebook_stage1_mlp",
#     "--epochs", "30",
# ])
# run_mlp(args)
